## BlueSky Sentiment Pipeline
**Topic:** BlueSky social posts → cleaned → AI-classified → visualised

**Stages:**
1. **Install** — one-time dependency setup  
2. **Extract** — authenticated BlueSky API pull  
3. **Transform** — dedup, filter, clean, timestamp  
4. **Load** — Gemini batch sentiment classification  
5. **Visualise** — sentiment distribution, daily trend, engagement by sentiment

> ⚠️ **Before running:** set your credentials in the *Configuration* cell below. Never commit API keys to version control.

## Configuration
Set all credentials and parameters here. This is the only cell you need to edit.

In [ ]:
# ── Configuration ── edit this cell only ────────────────────────────────
import os

# BlueSky credentials (use environment variables in production)
BSKY_HANDLE   = os.getenv("BSKY_HANDLE",   "")
BSKY_APP_PASS = os.getenv("BSKY_APP_PASS",  "")

# Google Gemini API key
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", "")

# Pipeline parameters
TOPIC        = "Ferrari F1"   # Search term
TARGET_POSTS = 550              # Approximate target post count
MIN_TEXT_LEN = 10               # Drop posts shorter than this
BATCH_SIZE   = 15               # Posts per Gemini API call
MODEL_ID     = "gemini-2.5-flash"

print("Configuration loaded.")
print(f"Topic: {TOPIC} | Target posts: {TARGET_POSTS} | Model: {MODEL_ID}")


Configuration loaded.
Topic: Ferrari F1 | Target posts: 550 | Model: gemini-2.5-flash


## Install Dependencies
Run once per environment. Safe to skip on subsequent runs.

In [2]:
# Install all required libraries in one cell
!pip install -q requests pandas plotly kaleido google-genai
print("Dependencies installed.")


Dependencies installed.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Extract — Pull Posts from BlueSky
Authenticates via  (avoids Cloudflare blocks on Colab IPs) and paginates the post search until  are collected.

In [3]:
import requests
import pandas as pd

# ── Authenticate ─────────────────────────────────────────────────────────
auth = requests.post(
    "https://bsky.social/xrpc/com.atproto.server.createSession",
    json={"identifier": BSKY_HANDLE, "password": BSKY_APP_PASS},
)
auth.raise_for_status()
access_token = auth.json()["accessJwt"]
print(f"Authenticated as: {auth.json()['handle']}")

# ── Paginated search ─────────────────────────────────────────────────────
HEADERS = {
    "Authorization": f"Bearer {access_token}",
    "Accept": "application/json",
}
API_URL   = "https://bsky.social/xrpc/app.bsky.feed.searchPosts"
all_posts = []
cursor    = None

print(f"Pulling posts about: {TOPIC}")

while len(all_posts) < TARGET_POSTS:
    params = {"q": TOPIC, "limit": 25}
    if cursor:
        params["cursor"] = cursor

    r = requests.get(API_URL, params=params, headers=HEADERS)

    if r.status_code != 200:
        print(f"API error {r.status_code}: {r.text[:200]}")
        break

    data  = r.json()
    posts = data.get("posts", [])
    if not posts:
        break

    for post in posts:
        record = post.get("record", {})
        author = post.get("author", {})
        all_posts.append({
            "post_id"   : post.get("uri", ""),
            "author"    : author.get("handle", ""),
            "text"      : record.get("text", ""),
            "created_at": record.get("createdAt", ""),
            "likes"     : post.get("likeCount", 0),
            "replies"   : post.get("replyCount", 0),
            "reposts"   : post.get("repostCount", 0),
        })

    cursor = data.get("cursor")
    if not cursor:
        break

df_raw = pd.DataFrame(all_posts)
df_raw.to_csv("raw_posts.csv", index=False)
print(f"Collected {len(df_raw)} posts → saved to raw_posts.csv")
print(df_raw[["author", "text", "created_at"]].head(3))


Authenticated as: forkyouabhi.bsky.social
Pulling posts about: Ferrari F1
Collected 563 posts → saved to raw_posts.csv
                      author  \
0  dailysportsfr.bsky.social   
1    ganegyzeten.bsky.social   
2       kentlive.bsky.social   

                                                text                created_at  
0  F1 : le changement majeur de Ferrari au GP de ...  2026-03-13T23:10:18.000Z  
1  MELLÉKHATÁS miatt vetették el a Ferrari gyrosn...  2026-03-13T23:01:42.791Z  
2  LEGO's Ferrari SF-24 F1 car is usually £23 and...  2026-03-13T22:47:58.831Z  


## Transform — Clean & Normalise
All cleaning in a single auditable cell:
- Deduplicate on 
- Drop posts shorter than  characters
- Parse timestamps (UTC-aware); drop rows with unparseable dates
- Extract  for daily trend grouping
- Normalise whitespace

In [4]:
import re

df_clean = df_raw.copy()

# 1. Deduplicate
n0 = len(df_clean)
df_clean = df_clean.drop_duplicates(subset="post_id")
n1 = len(df_clean)

# 2. Drop short posts
df_clean = df_clean[df_clean["text"].str.len() >= MIN_TEXT_LEN]
n2 = len(df_clean)

# 3. Parse timestamps (UTC-aware, drop failures)
df_clean["created_at"] = pd.to_datetime(df_clean["created_at"], utc=True, errors="coerce")
n_bad_ts = df_clean["created_at"].isna().sum()
df_clean = df_clean.dropna(subset=["created_at"])
n3 = len(df_clean)

# 4. Extract date component
df_clean["post_date"] = df_clean["created_at"].dt.date

# 5. Normalise whitespace
df_clean["text"] = df_clean["text"].str.replace(r"\s+", " ", regex=True).str.strip()

# 6. Reset index for downstream batch indexing
df_clean = df_clean.reset_index(drop=True)

df_clean.to_csv("clean_posts.csv", index=False)

print(f"Cleaning summary:")
print(f"  Raw posts          : {n0}")
print(f"  After dedup        : {n1}  (removed {n0 - n1})")
print(f"  After short filter : {n2}  (removed {n1 - n2})")
print(f"  After ts filter    : {n3}  (removed {n_bad_ts} bad timestamps)")
print(f"  Final clean posts  : {len(df_clean)}")
print(f"Saved to clean_posts.csv")


Cleaning summary:
  Raw posts          : 563
  After dedup        : 563  (removed 0)
  After short filter : 562  (removed 1)
  After ts filter    : 446  (removed 116 bad timestamps)
  Final clean posts  : 446
Saved to clean_posts.csv


## Load — AI Sentiment Classification (Gemini)
Sends posts to Gemini in batches of . Each post gets:
- **Sentiment** — Positive / Negative / Neutral  
- **Confidence** — High / Medium / Low  
- **Summary** — ≤15-word summary

Retries with exponential backoff on 429 rate-limit responses.

In [5]:
import json, time, requests as _req

GEMINI_URL = (
    f"https://generativelanguage.googleapis.com/v1beta/models/"
    f"{MODEL_ID}:generateContent"
)

# Inter-batch delay: pro models are heavily rate-limited on free tier
BATCH_DELAY = 65 if "pro" in MODEL_ID else 4

def classify_batch(posts_dict: dict, max_retries: int = 5) -> dict:
    """Send a batch of {id: text} posts to Gemini; return {id: classification}."""
    prompt = (
        "Classify the sentiment of the following social media posts. "
        "Respond ONLY with a valid JSON object where each key is the Post ID, "
        "and each value contains: 'Sentiment' (Positive | Negative | Neutral), "
        "'Confidence' (High | Medium | Low), and 'Summary' (≤15 words).\n\n"
        f"Posts: {json.dumps(posts_dict)}"
    )
    payload = {
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {
            "response_mime_type": "application/json",
            "temperature": 0,
        },
    }

    # 429s get their own unlimited retry loop — don't burn error retries on rate limits
    for attempt in range(max_retries):
        try:
            r = _req.post(GEMINI_URL, params={"key": GOOGLE_API_KEY}, json=payload, timeout=60)

            # --- Rate limit: keep retrying until the quota clears ---
            rate_attempt = 0
            while r.status_code == 429:
                wait = min(2 ** rate_attempt * 10, 120)   # 10, 20, 40, 80, 120s cap
                print(f"    Rate limited — waiting {wait}s (rate retry {rate_attempt+1})")
                time.sleep(wait)
                r = _req.post(GEMINI_URL, params={"key": GOOGLE_API_KEY}, json=payload, timeout=60)
                rate_attempt += 1

            r.raise_for_status()
            text = r.json()["candidates"][0]["content"]["parts"][0]["text"].strip()
            if text.startswith("```"):
                text = text.split("\n", 1)[-1]
                text = text.rsplit("```", 1)[0].strip()
            return json.loads(text)

        except Exception as e:
            print(f"    Batch error (attempt {attempt+1}): {e}")
            if attempt < max_retries - 1:
                time.sleep(5)

    return {}


# ── Batch loop ───────────────────────────────────────────────────────────
df_clean["sentiment"]  = "Error"
df_clean["confidence"] = "Low"
df_clean["summary"]    = ""

total = len(df_clean)
print(f"Classifying {total} posts with {MODEL_ID} (batch size={BATCH_SIZE}, delay={BATCH_DELAY}s)...\n")

for i in range(0, total, BATCH_SIZE):
    batch   = df_clean.iloc[i : i + BATCH_SIZE]
    batch_d = {str(idx): row["text"] for idx, row in batch.iterrows()}
    results = classify_batch(batch_d)

    for idx_str, result in results.items():
        idx = int(idx_str)
        df_clean.at[idx, "sentiment"]  = result.get("Sentiment",  "Neutral")
        df_clean.at[idx, "confidence"] = result.get("Confidence", "Medium")
        df_clean.at[idx, "summary"]    = result.get("Summary",    "")

    done = min(i + BATCH_SIZE, total)
    classified = (df_clean["sentiment"] != "Error").sum()
    print(f"  {done:>4} / {total} processed  |  {classified} classified so far")
    if i + BATCH_SIZE < total:
        time.sleep(BATCH_DELAY)

failed = (df_clean["sentiment"] == "Error").sum()
df_clean.to_csv("enriched_posts.csv", index=False)
print(f"\nDone. {total - failed} classified, {failed} errors. Saved enriched_posts.csv")


Classifying 446 posts with gemini-2.5-flash (batch size=15, delay=4s)...

    Rate limited — waiting 10s (rate retry 1)
    Rate limited — waiting 20s (rate retry 2)
    Rate limited — waiting 40s (rate retry 3)
    Rate limited — waiting 80s (rate retry 4)
    Rate limited — waiting 120s (rate retry 5)
    Rate limited — waiting 120s (rate retry 6)
    Rate limited — waiting 120s (rate retry 7)


KeyboardInterrupt: 

## 5️Visualise — Sentiment Analysis Charts
Three charts:
1. **Sentiment Distribution** — post count by sentiment category
2. **Daily Volume by Sentiment** — time-series stacked area
3. **Engagement by Sentiment** — average likes per sentiment

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd  # needed when running this cell independently

COLOR_MAP = {"Positive": "#2ECC71", "Neutral": "#95A5A6", "Negative": "#E74C3C"}

# Reload from file in case this cell is run independently
try:
    df_vis = df_clean.copy()
except NameError:
    df_vis = pd.read_csv("enriched_posts.csv", parse_dates=["created_at"])
    df_vis["post_date"] = pd.to_datetime(df_vis["created_at"]).dt.date

# Drop rows that failed classification so they don't pollute charts
df_vis = df_vis[df_vis["sentiment"].isin(["Positive", "Neutral", "Negative"])]

# ── Chart 1: Sentiment Distribution ──────────────────────────────────────
sent_counts = (
    df_vis["sentiment"].value_counts()
    .reindex(["Positive", "Neutral", "Negative"], fill_value=0)
    .reset_index()
)
sent_counts.columns = ["sentiment", "count"]

fig1 = go.Figure(go.Bar(
    x=sent_counts["sentiment"],
    y=sent_counts["count"],
    marker_color=[COLOR_MAP.get(s, "gray") for s in sent_counts["sentiment"]],
    text=sent_counts["count"],
    textposition="outside",
))
fig1.update_layout(
    title=f"Sentiment Distribution — {TOPIC} ({len(df_vis)} posts)",
    xaxis_title="Sentiment",
    yaxis_title="Post Count",
    plot_bgcolor="white",
    showlegend=False,
    margin=dict(t=80, l=60, r=40, b=60),
)
fig1.show()

# ── Chart 2: Daily Volume by Sentiment ────────────────────────────────────
daily = (
    df_vis.groupby(["post_date", "sentiment"])
    .size()
    .reset_index(name="count")
)
fig2 = px.area(
    daily,
    x="post_date",
    y="count",
    color="sentiment",
    color_discrete_map=COLOR_MAP,
    title=f"Daily Post Volume by Sentiment — {TOPIC}",
    labels={"post_date": "Date", "count": "Posts", "sentiment": "Sentiment"},
)
fig2.update_layout(plot_bgcolor="white")
fig2.show()

# ── Chart 3: Average Likes by Sentiment ───────────────────────────────────
eng = (
    df_vis.groupby("sentiment")["likes"]
    .mean()
    .reindex(["Positive", "Neutral", "Negative"])
    .reset_index()
)
fig3 = px.bar(
    eng,
    x="sentiment",
    y="likes",
    color="sentiment",
    color_discrete_map=COLOR_MAP,
    title=f"Average Likes by Sentiment — {TOPIC}",
    labels={"sentiment": "Sentiment", "likes": "Avg Likes"},
    text_auto=".1f",
)
fig3.update_layout(plot_bgcolor="white", showlegend=False)
fig3.show()


## Export

In [ ]:
# Works in both Jupyter (local) and Google Colab
try:
    from google.colab import files
    files.download("enriched_posts.csv")
    print("Colab download triggered.")
except ModuleNotFoundError:
    import os
    path = os.path.abspath("enriched_posts.csv")
    print(f"Running locally — file saved at:
  {path}")
